Project: /data-manager/api/_project.yaml
Book: /data-manager/api/_book.yaml

<style>
  devsite-code .tfo-notebook-code-cell-output {
    max-height: 300px;
    overflow: auto;
    background: rgba(255, 247, 237, 1);  /* light orange bg */
  }
  
  devsite-code .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
    background: rgba(255, 247, 237, .7);
  }
  
  devsite-code[dark-code] .tfo-notebook-code-cell-output {
    background: rgba(64, 78, 103, 1);  /* dark mode slate */
  }
  
  devsite-code[dark-code] .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
    background: rgba(64, 78, 103, .7);
  }
  
  .devsite-table-wrapper .tfo-notebook-buttons {
    display: inline-block;
    margin-left: 3px;
    width: auto;
    border: 0;
  }
  
  .tfo-notebook-buttons tr {
    background: 0;
    border: 0;
  }
  
  .tfo-notebook-buttons td {
    padding-left: 0;
    padding-right: 20px;
    border: 0;
  }
  
  .tfo-notebook-buttons {
    --tfo-notebook-buttons-box-shadow: 0 1px 2px 0 rgba(60, 64, 67, .3), 0 1px 3px 1px rgba(60, 64, 67, .15);
  }
  
  .tfo-notebook-buttons a,
  .tfo-notebook-buttons :link,
  .tfo-notebook-buttons :visited {
    border-radius: 8px;
    box-shadow: var(--tfo-notebook-buttons-box-shadow);
    color: #202124;
    padding: 12px 24px;
    transition: box-shadow 0.2s;
    text-decoration: none;
    display: flex;
    align-items: center;
  }
  
  .tfo-notebook-buttons a:hover,
  .tfo-notebook-buttons a:focus {
    box-shadow: 0 2px 6px 2px rgba(60, 64, 67, 0.15);
    text-decoration: none;
  }
  
  .tfo-notebook-buttons td > a > img {
    margin-right: 8px;
    width: 32px;
    height: 32px;
  }
  </style>

In [ ]:
# @markdown #### Copyright 2026 Google LLC
# @markdown ##### Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Data Partners: End-to-End Workflow with the Python SDK

  <table class="tfo-notebook-buttons nocontent" align="left">
    <td>
      <a target="_blank" href="https://colab.research.google.com/github/googleads/data-manager-python/blob/main/notebooks/data_partners_e2e_python_sdk.ipynb">
      <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
      Run in Google Colab</a>
    </td>
    <td>
      <a target="_blank" href="https://github.com/googleads/data-manager-python/blob/main/notebooks/data_partners_e2e_python_sdk.ipynb">
      <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
      View source on GitHub</a>
    </td>
  </table>

## Objective
This notebook provides a complete workflow for the Data Manager API: linking to a partner, creating a user list, ingesting data, and checking the status on the data ingestion.

**Key Goals:**
1. **Verify Google Cloud Configuration:** Ensure the Data Manager API and necessary scopes (for OAuth 2.0 or Service Accounts) are enabled.
2. **Prepare for App Verification (Milestone 2):** Provide a foundation for the app verification process. This is required when using OAuth workflows with the sensitive Data Manager API scope, applicable if you are making the partner linking experience available within your own app or platform.
3. **Facilitate API Explorer Testing:** For Steps 3 through 7, the notebook prints the required parameters and request payloads to replicate the calls in the Google API Explorer.

---

## The Two Personas
In a typical integration where partner linking happens within your platform's UI, two distinct roles interact with the API:
* **The Advertiser:** The Google Ads account owner who authorizes the connection using your UI (**Uses OAuth**).
* **The Data Partner (You):** The platform working in the background to retrieve the link, create audiences, and push data (**Uses a Service Account**).

## Choose Your Execution Path
Select the flow that matches your testing needs:

### Path A: Data Partner - Simplified Testing (Single Persona)
*Best for testing basic API mechanics or if you aren't hosting the linking experience in your UI.*
1. Ensure your authentication user (Service Account or OAuth) is an **ADMIN** on the linked Google Ads account.
2. Select your preferred authentication method in **Step 1**.
3. Run **Steps 1 through 6** sequentially.

### Path B: Data Partner - Simulating the In-Platform Experience (Dual Persona)
*Mimics the exact flow required for App Verification, where advertisers link to a partner within your UI.*

**Phase 1: Act as the Advertiser**
1. In **Step 1**, set `authentication_method` to **OAuth Client Credentials**.
2. Run **Steps 1 through 3** to consent and authorize the partner link.

**Phase 2: Act as the Data Partner**
3. Return to **Step 1** and change `authentication_method` to **Service Account**.
4. Run **Steps 1 and 2** to re-authenticate as your backend.
5. Skip Step 3, and resume from **Step 4** to retrieve the link and ingest data.

### Path C: Advertiser Flow
*Best for testing purely from the advertiser's perspective.*
1. Run **Steps 1 and 2**.
2. **Skip Step 3**.
3. Run **Steps 4 through 6**.
4. **Skip Step 7**.

---
## Instructions

1.  **Make a copy of this Colab:**
    *   Go to `File -> Save a copy in Drive`.

2.  **Choose your Authentication Method in the Setup cell below.**

3.  **If using OAuth Client Credentials:**
    *   Click the key icon in the left-hand sidebar.
    *   Create the following secrets and add your values:
        *   `CLIENT_ID`
        *   `CLIENT_SECRET`
        *   `REFRESH_TOKEN`
    *   Ensure "Notebook access" is enabled for each secret.

4. **If using a Service Account:**
    *   Fill in the `service_account_email` and `google_cloud_project` fields in the Setup cell.

### Step 0. Install Packages

In [ ]:
!pip install --upgrade google-ads-datamanager google-auth-oauthlib

### Step 1. Setup and Configuration

In [ ]:
import datetime
from google.colab import userdata
from google.ads import datamanager_v1
from google.oauth2.credentials import Credentials
import google.auth
from google.protobuf.json_format import MessageToJson

persona = "Advertiser"  # @param ["Data Partner", "Advertiser"]
authentication_method = "Service Account"  # @param ["OAuth Client Credentials", "Service Account"]

# @markdown ### Account IDs
# @markdown **DATA PARTNERS:** Provide the advertiser and data partner account IDs below.
# @markdown - **[required]** `linked_customer_id`: The Google Ads advertiser account.
# @markdown - **[required]** `login_customer_id`: Your Data Partner account.
# @markdown - (*optional*) `operating_account_id`: A specific sub-account to operate on. Leave blank if not using.

# @markdown **ADVERTISERS:**
# @markdown - `linked_customer_id`: Leave blank.
# @markdown - `login_customer_id`: Leave blank.
# @markdown - **[required]** `operating_account_id`: Your advertiser account

# @markdown **Enter Ids**
linked_customer_id = ""  # @param {type:"string"}
login_customer_id = ""  # @param {type:"string"}
operating_account_id = ""  # @param {type:"string"}

# @markdown ### Service Account Details (If applicable)
service_account_email = "" #@param {type:"string"}
google_cloud_project = "" #@param {type:"string"}

target_account_id = operating_account_id or linked_customer_id

GOOGLE_ADS = "GOOGLE_ADS"
DATA_PARTNER = "DATA_PARTNER"
CONSENT_GRANTED = "CONSENT_GRANTED"

### Step 2. Initialize Client

In [ ]:
# Create a wrapper to hold the three separate service clients
class DataManagerSDK:
    def __init__(self, creds):
        # There is no "DataManagerClient". You must use these three:
        self.link_service = datamanager_v1.PartnerLinkServiceClient(credentials=creds)
        self.user_list_service = datamanager_v1.UserListServiceClient(credentials=creds)
        self.ingestion_service = datamanager_v1.IngestionServiceClient(credentials=creds)

def initialize_client():
    if authentication_method == "OAuth Client Credentials":
        print("2. Authenticating with OAuth...")
        creds = Credentials(
            token=None,
            refresh_token=userdata.get('REFRESH_TOKEN'),
            client_id=userdata.get('CLIENT_ID'),
            client_secret=userdata.get('CLIENT_SECRET'),
            token_uri="https://oauth2.googleapis.com/token",
            scopes=["https://www.googleapis.com/auth/datamanager"]
        )
    else:
        print("2. Authenticating with Service Account...")
        auth_command = (
            f"gcloud auth application-default login "
            f"--impersonate-service-account={service_account_email} "
            f"--scopes=https://www.googleapis.com/auth/datamanager,https://www.googleapis.com/auth/cloud-platform"
        )
        get_ipython().system(auth_command)
        creds, project = google.auth.default()

    return DataManagerSDK(creds)

# Initialize
sdk = initialize_client()
print("SDK Services Initialized Successfully")

### Step 3. Create Partner Link

In [ ]:
print(f"3. Creating link for Ads {linked_customer_id}..\n")

parent_ads = f"accountTypes/GOOGLE_ADS/accounts/{linked_customer_id}"

partner_link_data = datamanager_v1.PartnerLink(
    owning_account=datamanager_v1.ProductAccount(
        account_id=linked_customer_id,
        account_type=GOOGLE_ADS
    ),
    partner_account=datamanager_v1.ProductAccount(
        account_id=login_customer_id,
        account_type=DATA_PARTNER
    )
)

try:
    link_res = sdk.link_service.create_partner_link(
        parent=parent_ads,
        partner_link=partner_link_data
    )
    print(f"Link Created: {link_res.partner_link_id}")
    product_link_id = link_res.partner_link_id
except Exception as e:
    print(f"Link creation failed or exists: {e}")
    print("\nSearching for existing link...")
    search_res = sdk.link_service.search_partner_links(parent=f"accountTypes/DATA_PARTNER/accounts/{login_customer_id}")
    # The search result is an iterator
    for link in search_res:
        if link.owning_account.account_id == linked_customer_id:
            product_link_id = link.partner_link_id
            print(f"Found existing Link ID: {product_link_id}")
            break

print("\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/accountTypes.accounts.partnerLinks/create?apix=true")
print(f"\nparent: {parent_ads}")
print("\n--- Request JSON Payload ---")
print(MessageToJson(partner_link_data._pb))
print("----------------------------\n")

### Step 4. Create User List

In [ ]:
print(f"4. Creating User List in {target_account_id}...\n")

parent_userlist = f"accountTypes/GOOGLE_ADS/accounts/{target_account_id}"

user_list_data = datamanager_v1.UserList(
    display_name=f"Python SDK Audience - {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    ingested_user_list_info=datamanager_v1.IngestedUserListInfo(
        upload_key_types=["CONTACT_ID"]
    )
)

# Start with the default arguments needed for both personas
call_kwargs = {
    "parent": parent_userlist,
    "user_list": user_list_data
}

# Only build and attach metadata if the persona is a Data Partner
if persona == "Data Partner":
    login_account = f"accountTypes/DATA_PARTNER/accounts/{login_customer_id}"
    linked_account = f"accountTypes/GOOGLE_ADS/accounts/{linked_customer_id}"

    call_kwargs["metadata"] =[
        ("login-account", login_account),
        ("linked-account", linked_account)
    ]

try:
    ulist_res = sdk.user_list_service.create_user_list(**call_kwargs)

    destination_id = ulist_res.id

    print(f"User List Created!")
    print(f"Destination ID: {destination_id}")
    print(f"Resource Name: {ulist_res.name}")
    print(f"Display Name: {ulist_res.display_name}")

except Exception as e:
    print(f"Failed to create User List: {e}")

print("\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/accountTypes.accounts.userLists/create?apix=true")
print(f"\nparent: {parent_userlist}")

if persona == "Data Partner":
    print(f"login-account: {login_account}")
    print(f"linked-account: {linked_account}")

print("\n--- Request JSON Payload ---")
print(MessageToJson(user_list_data._pb))
print("----------------------------\n")

### Step 5. Ingest Data

In [ ]:
print(f"5. Ingesting sample data to User List {destination_id}...")

# Build the Destination object explicitly based on the persona
if persona == "Data Partner":
    destination_obj = datamanager_v1.Destination(
        login_account=datamanager_v1.ProductAccount(
            account_type=DATA_PARTNER,
            account_id=login_customer_id
        ),
        operating_account=datamanager_v1.ProductAccount(
            account_type=GOOGLE_ADS,
            account_id=target_account_id
        ),
        linked_account=datamanager_v1.ProductAccount(
            account_type=GOOGLE_ADS,
            account_id=linked_customer_id
        ),
        product_destination_id=str(destination_id)
    )
elif persona == "Advertiser":
    # Advertiser Persona
    destination_obj = datamanager_v1.Destination(
        login_account=datamanager_v1.ProductAccount(
            account_type=GOOGLE_ADS,
            account_id=target_account_id # Advertiser login is their Google Ads account
        ),
        operating_account=datamanager_v1.ProductAccount(
            account_type=GOOGLE_ADS,
            account_id=target_account_id
        ),
        # Note: linked_account is omitted for Advertisers
        product_destination_id=str(destination_id)
    )
else:
    raise ValueError(
        f"Invalid persona: {persona}."
        f"Expected 'Data Partner' or 'Advertiser'."
    )

# Add the destination_obj directly to the payload
ingest_payload = datamanager_v1.IngestAudienceMembersRequest(
    consent=datamanager_v1.Consent(
        ad_user_data=CONSENT_GRANTED,
        ad_personalization=CONSENT_GRANTED
    ),
    encoding=datamanager_v1.Encoding.HEX,
    terms_of_service=datamanager_v1.TermsOfService(
        customer_match_terms_of_service_status="ACCEPTED"
    ),
    validate_only=False,
    audience_members=[
        datamanager_v1.AudienceMember(
            user_data=datamanager_v1.UserData(
                user_identifiers=[

                    # NOTE: email_address values must be hex-encoded SHA-256
                    # hashes of the normalized email addresses (lowercase, dots
                    # removed from gmail, etc.).

                    # Do NOT use plain text email addresses here.
                    # See Data Manager API documentation for formatting rules:
                    # https://developers.google.com/data-manager/api/devguides/concepts/formatting
                    datamanager_v1.UserIdentifier(email_address="223EBDA6F6889B1494551BA902D9D381DAF2F642BAE055888E96343D53E9F9C4"),
                    datamanager_v1.UserIdentifier(email_address="F1FCDE379F31F4D446B76EE8F34860ECA2288ADC6B6D6C0FDC56D9EEE75A2FA5")
                ]
            )
        )
    ],
    destinations=[destination_obj]
)

try:
    ingest_res = sdk.ingestion_service.ingest_audience_members(request=ingest_payload)

    ingestion_request_id = ingest_res.request_id
    print(f"Ingestion Submitted!")
    print(f"Request ID: {ingestion_request_id}")

except Exception as e:
    print(f"Failed to ingest data: {e}")
    ingestion_request_id = None

print("\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/audienceMembers/ingest?apix=true")
print("\n--- Request JSON Payload ---")
print(MessageToJson(ingest_payload._pb))
print("----------------------------\n")

### Step 6. Check Status

In [ ]:
if ingestion_request_id:
    print(f"6. Checking status for Request ID: {ingestion_request_id}...")

    status_req = datamanager_v1.RetrieveRequestStatusRequest(
        request_id=ingestion_request_id
    )

    try:
        status_res = sdk.ingestion_service.retrieve_request_status(request=status_req)

        print(f"Successfully retrieved status!")

        for dest_status in status_res.request_status_per_destination:
            dest_id = dest_status.destination.product_destination_id
            current_status = dest_status.request_status.name

            print(f"Status for destination {dest_id}: {current_status}")

    except Exception as e:
        print(f"Failed to retrieve status: {e}")

print("\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/requestStatus/retrieve?apix=true")
print("\n--- Request JSON Payload ---")
print(MessageToJson(status_req._pb))
print("----------------------------")


### Step 7. [Optional] Remove Product Link

In [ ]:
if product_link_id:
    print(f"\nAttempting to remove product link {product_link_id}...")

    link_resource_name = f"accountTypes/GOOGLE_ADS/accounts/{linked_customer_id}/partnerLinks/{product_link_id}"

    delete_req = datamanager_v1.DeletePartnerLinkRequest(
        name=link_resource_name
    )

    try:
        sdk.link_service.delete_partner_link(request=delete_req)
        print("Successfully removed product link.")

    except Exception as e:
        print(f"Failed to remove product link: {e}")
else:
    print("\nNo product_link_id found to delete. Did you run Step 1?")

print("\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/accountTypes.accounts.partnerLinks/delete?apix=true")
print("\n--- Request JSON Payload ---")
print(MessageToJson(delete_req._pb))
print("----------------------------")
